# Notebook 07 — Acute Oral Toxicity LD50 Prediction
**Author: Himanshu Goel** | [Website](https://himanshugoel.github.io)

LD50 (lethal dose, 50%) is the primary acute systemic toxicity measure. Prediction is required for GHS classification, REACH registration, and chemical inventory assessments.

| GHS Category | LD50 oral rat (mg/kg) | Signal word |
|---|---|---|
| Cat 1 | <= 5 | Danger |
| Cat 2 | 5-50 | Danger |
| Cat 3 | 50-300 | Danger |
| Cat 4 | 300-2000 | Warning |
| Cat 5 | 2000-5000 | (May be harmful) |

Key dataset: Zhu et al. 2009 (7413 compounds) — still the benchmark for LD50 QSAR.

In [ ]:
!pip install rdkit scikit-learn xgboost pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import warnings; warnings.filterwarnings('ignore')

# LD50 dataset (rat oral, mg/kg) — representative from Zhu 2009 / ChemIDplus
ld50_data = [
    ("CC(=O)Nc1ccc(O)cc1",  2404, "Acetaminophen"),
    ("CC(=O)O",             3310, "Acetic acid"),
    ("ClC(Cl)(Cl)Cl",       1770, "CCl4"),
    ("CN(C)C(=N)NC(=N)N",  3000, "Metformin"),
    ("OC(=O)c1ccccc1",     2530, "Benzoic acid"),
    ("c1ccc(Cl)c(Cl)c1",    500, "1,2-DCB"),
    ("O=Cc1ccccc1",         1300, "Benzaldehyde"),
    ("CC(C)=O",             5800, "Acetone"),
    ("OCC(O)CO",           27200, "Glycerol"),
    ("[O-][N+](=O)c1ccccc1", 489, "Nitrobenzene"),
    ("c1ccc2[nH]ccc2c1",   1000, "Indole"),
    ("C1CCCCC1",           12705, "Cyclohexane"),
    ("CC(C)Cc1ccc(cc1)C(C)C(=O)O",1255,"Ibuprofen"),
    ("c1ccc(NN)cc1",          80, "Phenylhydrazine"),
    ("Nc1ccccc1",             440, "Aniline"),
    ("OCC(O)C(O)C(O)CO",  15900, "Xylitol"),
    ("OC(=O)CS",            1600, "Thioglycolic acid"),
    ("CC(=O)OCC",           5620, "Ethyl acetate"),
    ("Cc1ccc(S(=O)(=O)Nc2ccccn2)cc1",842,"Sulfadiazine"),
    ("CC(C)(C)c1ccc(O)cc1",3250, "4-tBu-phenol"),
]

def ghs_cat(ld50):
    if ld50<=5: return 1
    elif ld50<=50: return 2
    elif ld50<=300: return 3
    elif ld50<=2000: return 4
    else: return 5

def featurize(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    fp=np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,2048))
    pc=np.array([
        Descriptors.ExactMolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol), Descriptors.FractionCSP3(mol),
        Descriptors.MolMR(mol), rdMolDescriptors.CalcNumRings(mol),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==16),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in [9,17,35,53]),
        Descriptors.NumValenceElectrons(mol),
    ])
    return np.concatenate([fp,pc])

valid=[(s,ld,n) for s,ld,n in ld50_data if featurize(s) is not None]
X=np.array([featurize(s) for s,_,_ in valid])
y=np.log10([ld for _,ld,_ in valid])
names=[n for _,_,n in valid]
cats=np.array([ghs_cat(ld) for _,ld,_ in valid])
scaler=StandardScaler(); X_s=scaler.fit_transform(X)
print(f"LD50 dataset: {len(y)} | log10 range: {y.min():.2f}-{y.max():.2f}")
print(f"GHS distribution: {dict(zip(*np.unique(cats,return_counts=True)))}")

In [ ]:
kf=KFold(4,shuffle=True,random_state=42)
print("LD50 Regression (log10 mg/kg):")
print(f"{'Model':20s} {'R2':>8} {'RMSE':>10}")
for nm,reg in [("Random Forest",RandomForestRegressor(300,random_state=42)),
               ("XGBoost",XGBRegressor(200,random_state=42,verbosity=0)),
               ("GradBoost",GradientBoostingRegressor(200,random_state=42))]:
    r2=cross_val_score(reg,X_s,y,cv=kf,scoring='r2')
    mse=cross_val_score(reg,X_s,y,cv=kf,scoring='neg_mean_squared_error')
    print(f"{nm:20s} {r2.mean():8.3f} {(-mse.mean())**0.5:10.3f}")

rf=RandomForestRegressor(300,random_state=42).fit(X_s,y)
yp=rf.predict(X_s)
ghs_col={1:'#7d0000',2:'#e74c3c',3:'#f39c12',4:'#f1c40f',5:'#27ae60'}
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(13,5))
for cat in sorted(ghs_col):
    mask=cats==cat
    ax1.scatter(y[mask],yp[mask],c=ghs_col[cat],s=70,label=f"GHS Cat {cat}",alpha=0.85)
for n,yt,ypr in zip(names,y,yp):
    ax1.annotate(n,(yt,ypr),fontsize=6,xytext=(2,2),textcoords='offset points')
xl=np.linspace(y.min()-0.2,y.max()+0.2,100)
ax1.plot(xl,xl,'k--',lw=1); ax1.set_xlabel("log10(LD50) Observed"); ax1.set_ylabel("Predicted")
ax1.set_title("LD50 Regression"); ax1.legend(fontsize=8)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
pred_cats=np.array([ghs_cat(10**p) for p in yp])
cm=confusion_matrix(cats,pred_cats,labels=[1,2,3,4,5])
ConfusionMatrixDisplay(cm,display_labels=[f"Cat{i}" for i in range(1,6)]).plot(
    ax=ax2,colorbar=False,cmap='Blues')
ax2.set_title("GHS Category Confusion Matrix")
plt.tight_layout(); plt.savefig("ld50_ghs.png",dpi=150); plt.show()
print(f"GHS accuracy: {(cats==pred_cats).mean():.1%}")

## Key takeaways
- log10 transformation is mandatory for LD50 regression (skewed 4-log distribution)
- GHS category accuracy is the most practically useful metric for regulatory classification
- RMSE ~ 0.5 log-units = 3-fold error in LD50 — typical state-of-the-art performance
- Industry datasets: Zhu 2009 (7413 cpds), ToxValDB (EPA), OpenTox, ChemIDplus
- OECD tools: T.E.S.T., EPA OPERA, QSAR Toolbox — all use similar feature spaces
- 3Rs alignment: LD50 models reduce animal testing per OECD TG 423/425